In [1]:
import os
import pandas as pd
import altair as alt
import numpy as np
os.chdir('/home/jupyter/COGS 402/ai_semiconductor_project')

# **Topic**: AI Semiconductors

## **Dataset 1**: Kaggle – Open Database on Global Coal and Metal Mining

### **1. material_ids.csv**

In [2]:
material_ids_df = pd.read_csv('data/raw/dataset1/material_ids.csv')

# search for Silicon (Si)
silicon = material_ids_df[material_ids_df['material_name'].str.contains("Silicon")]
print("Silicon (Si):")
print(silicon[['material_id', 'material_name']])

# search for Germanium (Ge)
germanium = material_ids_df[material_ids_df['material_name'].str.contains("Germanium")]
print("\nGermanium (Ge):")
print(germanium[['material_id', 'material_name']])

# search for Gallium Arsenide (GaAs)
# search for Gallium (Ga)
gallium = material_ids_df[material_ids_df['material_name'].str.contains("Gallium")]
print("\nGallium (Ga):")
print(gallium[['material_id', 'material_name']])

# search for Arsenide (As)
# since Arsenide is a byproduct of copper mining, and it is absent in this dataset, I use "Copper dominant ore" to represent
arsenide = material_ids_df[material_ids_df['material_name'].str.contains("Copper dominant ore")]
print("\nArsenide (As):")
print(arsenide[['material_id', 'material_name']])

Silicon (Si):
    material_id material_name
199       Me.Si       Silicon

Germanium (Ge):
    material_id material_name
131       Me.Ge     Germanium

Gallium (Ga):
    material_id material_name
130       Me.Ga       Gallium

Arsenide (As):
  material_id        material_name
7        O.Cu  Copper dominant ore


### **2. commodities.csv**

In [3]:
commodities_df = pd.read_csv('data/raw/dataset1/commodities.csv')
print(f"dataset shape: {commodities_df.shape}")
commodities_df.info()

dataset shape: (9426, 16)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9426 entries, 0 to 9425
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    9426 non-null   int64  
 1   facility_id           9426 non-null   object 
 2   year                  9426 non-null   int64  
 3   material              8895 non-null   object 
 4   commodity             9426 non-null   object 
 5   value_tonnes          9399 non-null   float64
 6   grade_ppm             7132 non-null   float64
 7   recovery_rate         5497 non-null   float64
 8   yield_ppm             127 non-null    float64
 9   amount_sold_tonnes    1423 non-null   float64
 10  metal_payable_tonnes  429 non-null    float64
 11  mine_processing       473 non-null    object 
 12  reporting_period      1233 non-null   object 
 13  source_id             9426 non-null   object 
 14  comment               386 non-null    object 


In [4]:
raw_materials = {
    'Me.Si': 'Silicon',
    'Me.Ge': 'Germanium', 
    'Me.Ga': 'Gallium',
    'O.Cu': 'Copper_dominant_ore'  # represents arsenic-bearing copper ore
}

commodities_clean_df = commodities_df[
    (commodities_df['commodity'].isin(raw_materials.keys())) & (commodities_df['value_tonnes'].notna())
].copy()

print(f"Original data: {len(commodities_df)} rows")
print(f"Filtered data: {len(commodities_clean_df)} rows")

Original data: 9426 rows
Filtered data: 0 rows


### **3. ownership.csv**

In [5]:
ownership_df = pd.read_csv('data/raw/dataset1/ownership.csv')
print(f"dataset shape: {ownership_df.shape}")
ownership_df.info()

dataset shape: (699, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 699 entries, 0 to 698
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           699 non-null    int64 
 1   facility_id  699 non-null    object
 2   year         699 non-null    int64 
 3   owners       698 non-null    object
 4   operators    143 non-null    object
 5   source_id    699 non-null    object
 6   comment      8 non-null      object
dtypes: int64(2), object(5)
memory usage: 38.4+ KB


In [6]:
interest_cols = ['facility_id', 'year', 'owners']
valid_rows = (ownership_df['owners'].notna()) & (ownership_df['year'] >= 2015)
ownership_clean = ownership_df[valid_rows][interest_cols].copy()

print(f"dataset shape: {ownership_clean.shape}")
ownership_clean.sample(5)

dataset shape: (630, 3)


,facility_id,year,owners
87,COM00186.00,2019,Pangea Minerals Limited (PML) subsidiary of Ac...
477,COM00971.00,2019,Newmont Mining Corp. (100%)
105,COM00222.00,2018,MetInvest B.V. (100%)
27,COM00067.00,2018,Rio Tinto (100%)
536,COM01098.00,2018,Hudbay Minerals Inc (100%)


### **Dataset 2**: USGS – U.S. Geological Survey Mineral Commodity Summaries 2025 Data Release

In [7]:
silicon_df = pd.read_csv('data/raw/dataset2/mcs2025-simet_salient.csv')
germanium_df = pd.read_csv('data/raw/dataset2/mcs2025-germa_salient.csv')
gallium_df = pd.read_csv('data/raw/dataset2/mcs2025-galli_salient.csv')
arsenic_df = pd.read_csv('data/raw/dataset2/mcs2025-arsen_salient.csv')

In [8]:
prices_clean = pd.DataFrame({
    'Year': silicon_df['Year'],
    'Silicon_cents_per_lb': silicon_df['Price_Si_ctslb'],
    'Germanium_USD_per_kg': germanium_df['Price_Metal_dkg'],
    'Gallium_USD_per_kg': gallium_df['Price_High-purity_dkg'],
    'Arsenic_USD_per_kg': arsenic_df['Price_Mtl_US_dkg']
})

prices_clean.to_csv('data/processed/dataset2/prices_clean.csv', index = False)
print(f"dataset shape: {prices_clean.shape}")
prices_clean

dataset shape: (5, 5)


,Year,Silicon_cents_per_lb,Germanium_USD_per_kg,Gallium_USD_per_kg,Arsenic_USD_per_kg
0,2020,96.84,1046,596,1.08
1,2021,220.31,1187,625,1.11
2,2022,361.86,1294,560,1.67
3,2023,179.69,1392,450,2.05
4,2024,180.00,2100,500,2.00


### **Dataset 3**: Global Trade Alert – Export Controls on Semiconductors

In [9]:
interventions_df = pd.read_csv('data/raw/dataset3/interventions.csv')
print(f"dataset shape: {interventions_df.shape}")
interventions_df.info()

dataset shape: (80, 19)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   Intervention ID             80 non-null     int64 
 1   Intervention URL            80 non-null     object
 2   State Act ID                80 non-null     int64 
 3   State Act URL               80 non-null     object
 4   State Act Title             80 non-null     object
 5   GTA Evaluation              80 non-null     object
 6   Implementing Jurisdictions  80 non-null     object
 7   Implementation Level        80 non-null     object
 8   Eligible Firm               80 non-null     object
 9   Intervention Type           80 non-null     object
 10  Mast Chapter                80 non-null     object
 11  Affected Sectors            79 non-null     object
 12  Affected Products           79 non-null     object
 13  Affected Jurisdictions      

In [10]:
interventions_clean = interventions_df[
    interventions_df['Is In Force'] == 1
][['State Act Title', 'Implementing Jurisdictions', 'Implementation Level', 'Intervention Type', 'Affected Products', 'Affected Jurisdictions', 'Date Implemented', 'Is In Force']
    ].dropna(subset = ['Affected Products', 'Affected Jurisdictions']).copy()

interventions_clean['Date Implemented'] = pd.to_datetime(interventions_clean['Date Implemented'], errors = 'coerce')
interventions_clean['Year'] = interventions_clean['Date Implemented'].dt.year

interventions_clean.to_csv('data/processed/dataset3/interventions_clean.csv', index = False)
print(f"dataset shape: {interventions_clean.shape}")
interventions_clean.sample(5)

dataset shape: (74, 9)


,State Act Title,Implementing Jurisdictions,Implementation Level,Intervention Type,Affected Products,Affected Jurisdictions,Date Implemented,Is In Force,Year
37,EU: 11th sanctions package against Russia incl...,"Austria, Belgium, Bulgaria, Croatia, Cyprus, C...",Supranational,Export ban,"240412, 240492, 261210, 261220, 262099, 270710...",United Arab Emirates,2023-06-24,1,2023
60,United States of America: Government restricts...,United States of America,National,Export licensing requirement,"854151, 854159, 854231, 854232, 854233, 854239...",China,2024-11-10,1,2024
68,United States of America: Imposition of export...,United States of America,National,Export licensing requirement,"852349, 852351, 854231, 854232, 854233, 854239","Afghanistan, Albania, Algeria, American Samoa,...",2025-05-13,1,2025
66,EU: 15th sanctions package against Russia targ...,"Austria, Belgium, Bulgaria, Croatia, Cyprus, C...",Supranational,Export ban,"261210, 261220, 262099, 270710, 270720, 270730...","China, Hong Kong, Iran, Russia, Serbia, India,...",2024-12-17,1,2024
51,United States of America: Commerce Department ...,United States of America,National,Export licensing requirement,"261210, 261220, 262099, 270710, 270720, 270730...","Armenia, Belgium, Belarus, China, Cyprus, Germ...",2023-12-06,1,2023


### **Dataset 4**: Emerging Technology Observatory – Advanced Semiconductor Supply Chain Dataset

### **1. inputs.csv**

In [11]:
inputs_df = pd.read_csv('data/raw/dataset4/inputs.csv')
print(f"dataset shape: {inputs_df.shape}")
inputs_df.info()

dataset shape: (126, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 126 entries, 0 to 125
Data columns (total 10 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   input_id                                    126 non-null    object 
 1   input_name                                  126 non-null    object 
 2   type                                        126 non-null    object 
 3   stage_name                                  11 non-null     object 
 4   stage_id                                    11 non-null     object 
 5   description                                 126 non-null    object 
 6   year                                        104 non-null    float64
 7   market_share_chart_global_market_size_info  103 non-null    object 
 8   market_share_chart_caption                  0 non-null      float64
 9   market_share_chart_source                   103 non-null    ob

In [12]:
inputs_clean = inputs_df[['input_name', 'type', 'stage_name', 'year', 'market_share_chart_global_market_size_info']
    ].dropna(subset = ['year']).copy()

inputs_clean.to_csv('data/processed/dataset4/inputs_clean.csv', index = False)
print(f"dataset shape: {inputs_clean.shape}")
inputs_clean.sample(5)

dataset shape: (104, 5)


,input_name,type,stage_name,year,market_share_chart_global_market_size_info
37,High-temperature CVD tools,tool_resource,NaN,2024.0,$1.5 billion (2024)
63,"Film, stack, and shape metrology tools",tool_resource,NaN,2024.0,$2.3 billion (2024)
62,Process monitoring tools,tool_resource,NaN,2024.0,$50.9 million (2024)
29,Photoresists,material_resource,NaN,2019.0,$3.3 billion (all photoresists)
42,Electrochemical plating tools,tool_resource,NaN,2024.0,$688.7 million (2024)


### **2. providers.csv**

In [13]:
providers_df = pd.read_csv('data/raw/dataset4/providers.csv')
print(f"dataset shape: {providers_df.shape}")
providers_df.info()

dataset shape: (397, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397 entries, 0 to 396
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   provider_name  397 non-null    object
 1   alias          68 non-null     object
 2   provider_id    397 non-null    object
 3   provider_type  397 non-null    object
 4   country        375 non-null    object
dtypes: object(5)
memory usage: 15.6+ KB


In [14]:
providers_clean = providers_df[['provider_name', 'provider_type', 'country']].dropna(subset = ['country']).copy()

providers_clean.to_csv('data/processed/dataset4/providers_clean.csv', index = False)
print(f"dataset shape: {providers_clean.shape}")
providers_clean.sample(5)

dataset shape: (375, 3)


,provider_name,provider_type,country
374,Royce Instruments,organization,USA
121,GigaLane,organization,KOR
73,ESI,organization,USA
176,Synopsys,organization,USA
125,KCTech,organization,KOR


### **3. provision.csv**

In [15]:
provision_df = pd.read_csv('data/raw/dataset4/provision.csv')
print(f"dataset shape: {provision_df.shape}")
provision_df.info()

dataset shape: (1305, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1305 entries, 0 to 1304
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   provider_name   1305 non-null   object 
 1   provider_id     1305 non-null   object 
 2   provided_name   1297 non-null   object 
 3   provided_id     1305 non-null   object 
 4   share_provided  1051 non-null   float64
 5   year            1305 non-null   int64  
 6   source          1170 non-null   object 
dtypes: float64(1), int64(1), object(5)
memory usage: 71.5+ KB


In [16]:
provision_clean = provision_df[['provider_name', 'provided_name', 'share_provided', 'year']
    ].dropna(subset = ['provided_name', 'share_provided']).copy()

provision_clean.to_csv('data/processed/dataset4/provision_clean.csv', index = False)
print(f"dataset shape: {provision_clean.shape}")
provision_clean.sample(5)

dataset shape: (1051, 4)


,provider_name,provided_name,share_provided,year
639,SEMES,Etch and clean tools,1.8,2024
423,KLA,Photomask inspection and repair tools,40.7,2024
503,Lam Research,Deposition tools (adv. pkg.),29.4,2024
320,ISR,Fabrication tools (for advanced packaging),2.9,2024
741,Thermco Epitaxy,High-temperature CVD tools,1.3,2024


### **4. sequence.csv**

In [17]:
sequence_df = pd.read_csv('data/raw/dataset4/sequence.csv')
print(f"dataset shape: {sequence_df.shape}")
sequence_df.info()

dataset shape: (139, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 139 entries, 0 to 138
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   input_name       139 non-null    object
 1   input_id         139 non-null    object
 2   goes_into_name   53 non-null     object
 3   goes_into_id     53 non-null     object
 4   is_type_of_name  86 non-null     object
 5   is_type_of_id    86 non-null     object
dtypes: object(6)
memory usage: 6.6+ KB


In [18]:
sequence_clean = sequence_df[['input_name', 'goes_into_name', 'is_type_of_name']].copy()

sequence_clean.to_csv('data/processed/dataset4/sequence_clean.csv', index = False)
print(f"dataset shape: {sequence_clean.shape}")
sequence_clean.sample(5)

dataset shape: (139, 3)


,input_name,goes_into_name,is_type_of_name
102,DAO chip design,Chip design,NaN
58,Burn-in test tools,NaN,Test tools
61,Core intellectual property,EDA and Core IP,NaN
99,Spin-on deposition tools (adv. pkg.),NaN,Deposition tools (adv. pkg.)
20,Rapid thermal processing tools,NaN,Deposition tools


### **Dataset 5**: FRED – Producer Price Index by Industry: Semiconductor and Other Electronic Component Manufacturing

In [19]:
chip_price_df = pd.read_csv('data/raw/dataset5/PCU33443344.csv')
print(f"dataset shape: {chip_price_df.shape}")
chip_price_df.info()

dataset shape: (493, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 493 entries, 0 to 492
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   observation_date  493 non-null    object 
 1   PCU33443344       493 non-null    float64
dtypes: float64(1), object(1)
memory usage: 7.8+ KB


In [20]:
chip_price_clean = chip_price_df.copy()
chip_price_clean['observation_date'] = pd.to_datetime(chip_price_clean['observation_date'])
chip_price_clean['Year'] = chip_price_clean['observation_date'].dt.year
chip_price_clean['Month'] = chip_price_clean['observation_date'].dt.month
chip_price_clean.rename(columns = {
    'PCU33443344': 'Price_Index'
}, inplace = True)

chip_price_clean = chip_price_clean[chip_price_clean['Year'] >= 2015].copy()

chip_price_clean.to_csv('data/processed/dataset5/chip_price_clean.csv', index = False)
print(f"dataset shape: {chip_price_clean.shape}")
chip_price_clean.sample(5)

dataset shape: (132, 4)


,observation_date,Price_Index,Year,Month
488,2025-08-01,59.626,2025,8
361,2015-01-01,58.700,2015,1
412,2019-04-01,55.200,2019,4
456,2022-12-01,57.607,2022,12
393,2017-09-01,56.000,2017,9


### **Dataset 6**: OECD – Analytical Business Enterprise R&D by ISIC Rev.4 industry

In [21]:
oecd_df = pd.read_csv('data/raw/dataset6/OECD.csv')
print(f"dataset shape: {oecd_df.shape}")
oecd_df.info()

dataset shape: (62, 30)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62 entries, 0 to 61
Data columns (total 30 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   STRUCTURE                 62 non-null     object 
 1   STRUCTURE_ID              62 non-null     object 
 2   STRUCTURE_NAME            62 non-null     object 
 3   ACTION                    62 non-null     object 
 4   REF_AREA                  62 non-null     object 
 5   Reference area            62 non-null     object 
 6   FREQ                      62 non-null     object 
 7   Frequency of observation  62 non-null     object 
 8   CRITERIA                  62 non-null     object 
 9   Classification criteria   62 non-null     object 
 10  ACTIVITY                  62 non-null     object 
 11  Economic activity         62 non-null     object 
 12  UNIT_MEASURE              62 non-null     object 
 13  Unit of measure           62 non-null     o

In [22]:
oecd_clean = oecd_df[['Reference area', 'Economic activity', 'TIME_PERIOD', 'OBS_VALUE', 'Unit of measure']].copy()
oecd_clean.rename(columns = {
    'Reference area': 'Country',
    'TIME_PERIOD': 'Year',
    'OBS_VALUE': 'Investment amount'
}, inplace = True)

oecd_clean.to_csv('data/processed/dataset6/oecd_investment_clean.csv', index = False)
print(f"dataset shape: {oecd_clean.shape}")
oecd_clean.sample(5)

dataset shape: (62, 5)


,Country,Economic activity,Year,Investment amount,Unit of measure
0,Korea,Manufacture of electronic components and boards,2021,11914575000000,National currency
37,Germany,"Manufacture of computer, electronic and optica...",2018,8280600000,National currency
3,Germany,"Manufacture of computer, electronic and optica...",2021,9034391000,National currency
44,Korea,Manufacture of electronic components and boards,2018,9011165000000,National currency
31,United States,Manufacture of electronic components and boards,2020,43952000000,National currency


### **Dataset 7**: European Commission – The 2025 EU Industrial R&D Investment Scoreboard

In [23]:
eu_scoreboard_df = pd.read_excel('data/raw/dataset7/SB2025_EU800.xlsx')
eu_scoreboard_df.to_csv('data/raw/dataset7/SB2025_EU800.csv', index = False)
print(f"dataset shape: {eu_scoreboard_df.shape}")
eu_scoreboard_df.info()

dataset shape: (800, 24)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 24 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Year                                       800 non-null    int64  
 1   World rank                                 318 non-null    float64
 2   EU rank                                    800 non-null    int64  
 3   Company                                    800 non-null    object 
 4   Country                                    800 non-null    object 
 5   ISO3 Code                                  800 non-null    object 
 6   ICB3 short                                 800 non-null    object 
 7   ICB3 long                                  800 non-null    object 
 8   R&D (€million)                             800 non-null    float64
 9   R&D one-year growth (%)                    768 non-null    float64
 10  N

In [24]:
semiconductor_sectors = [
    'Technology Hardware & Equipment',
    'Electronic & Electrical Equipment',
    'Software & Computer Services'
]

eu_scoreboard_clean = eu_scoreboard_df[
    eu_scoreboard_df['ICB3 long'].isin(semiconductor_sectors)
][[
    'Year',
    'Company',
    'Country',
    'ICB3 long',
    'R&D (€million)',
    'R&D one-year growth (%)',
    'R&D intensity (%)',
    'Capex (€million)',
    'Capex one-year growth (%)',
    'Capex intensity (%)',
    'R&D-to-capex ratio (%)',
    'Net sales (€million)',
    'Net sales one-year growth (%)',
    'Operating profit (€million)',
    'Operating profit one-year growth (%)',
    'Profitability (%)',
    'Market capitalisation (€million)',
    'Market capitalisation one-year growth (%)',
    'Employees',
    'Employees one-year growth (%)'
]].dropna(subset = ['Market capitalisation (€million)']).copy()

eu_scoreboard_clean.to_csv('data/processed/dataset7/eu_scoreboard_clean.csv', index = False)
print(f"dataset shape: {eu_scoreboard_clean.shape}")
eu_scoreboard_clean.sample(5)

dataset shape: (129, 20)


,Year,Company,Country,ICB3 long,R&D (€million),R&D one-year growth (%),R&D intensity (%),Capex (€million),Capex one-year growth (%),Capex intensity (%),R&D-to-capex ratio (%),Net sales (€million),Net sales one-year growth (%),Operating profit (€million),Operating profit one-year growth (%),Profitability (%),Market capitalisation (€million),Market capitalisation one-year growth (%),Employees,Employees one-year growth (%)
202,2024,SINCH AB,Sweden,Software & Computer Services,149.140414,32.583398,5.952215,16.668121,-9.478673,0.665227,894.764398,2505.628763,-0.114803,-510.603019,-961.708395,-20.378239,1527.766236,-42.829392,4077.0,-3.639801
572,2024,VERIMATRIX,France,Technology Hardware & Equipment,18.994128,-5.646935,34.495236,0.096256,-68.944099,0.174810,19733.000000,55.063047,-7.185969,-3.737607,-53.301263,-6.787868,24.206446,-41.540800,253.0,2.845528
677,2024,NET INSIGHT AB,Sweden,Technology Hardware & Equipment,11.024522,-7.607583,20.777585,0.425953,87.947632,0.802782,2588.199140,53.059691,8.696064,6.954184,4.721729,13.106342,228.183038,40.794145,154.0,5.479452
789,2024,ACCONEER AB,Sweden,Technology Hardware & Equipment,5.867353,NaN,131.009353,0.447945,NaN,10.001949,1309.838301,4.478576,NaN,-2.699014,NaN,-60.265004,26.035803,NaN,56.0,NaN
7,2024,SIEMENS AG,Germany,Electronic & Electrical Equipment,6482.000000,-0.030845,8.224740,2088.000000,-5.861136,2.649376,310.440613,78811.000000,1.339865,9608.000000,3.948934,12.191192,145600.000000,34.515888,327000.0,2.187500


### **Dataset 8**: Kaggle – Top Semiconductor Companies by Market Capital

In [25]:
sc_data_df = pd.read_csv('data/raw/dataset8/sc_data.csv')
print(f"dataset shape: {sc_data_df.shape}")
sc_data_df.info()

dataset shape: (152, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 152 entries, 0 to 151
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   rank                   152 non-null    int64  
 1   name                   152 non-null    object 
 2   ticker                 152 non-null    object 
 3   market_cap_04_08_2025  152 non-null    int64  
 4   price_04_08_2025       152 non-null    float64
 5   country                152 non-null    object 
dtypes: float64(1), int64(2), object(3)
memory usage: 7.3+ KB


In [26]:
sc_data_clean = sc_data_df[['rank', 'name', 'ticker', 'country', 'market_cap_04_08_2025', 'price_04_08_2025']].copy()

sc_data_clean.rename(columns = {
    'name': 'company',
    'market_cap_04_08_2025': 'market_cap_usd',
    'price_04_08_2025': 'stock_price_usd'
}, inplace = True)
sc_data_clean['Data_Date'] = '2025-04-08'

sc_data_clean.to_csv('data/processed/dataset8/semiconductor_companies_clean.csv', index = False)
print(f"dataset shape: {sc_data_clean.shape}")
sc_data_clean.sample(5)

dataset shape: (152, 7)


,rank,company,ticker,country,market_cap_usd,stock_price_usd,Data_Date
21,22,NXP Semiconductors,NXPI,Netherlands,52923981824,209.92000,2025-04-08
57,58,Lattice Semiconductor,LSCC,United States,6736517120,48.99000,2025-04-08
100,101,ACM Research,ACMR,United States,1882725376,29.48000,2025-04-08
142,143,centrotherm international,CTNK.F,Germany,121973486,5.76369,2025-04-08
119,120,AT&amp;S Austria Technologie &amp; Systemtechnik,ATS.VI,Austria,789987295,20.33430,2025-04-08
